[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jgcalvo/SP1306/blob/main/edps-parabolicas/MEF_calor2D.ipynb)

# Ecuación del calor en 2D con elementos P1 y el método $\theta$

Traducción de `MEF_calor2D.m`. Es la versión en 2D de `MEF_calor1D.ipynb`.

$$u_t = \Delta u \quad \text{en } \Omega\times(0,T], \quad \Omega=(0,1)^2,
\qquad u = 0 \ \text{ sobre } \partial\Omega, \qquad u(x,y,0)=u_0(x,y)$$

## Formulación débil

Multiplicando por $v$ que se anula en la frontera e integrando por partes el
término de segundo orden,

$$(u_t, v) + (\nabla u, \nabla v) = (f, v) \qquad \forall v$$

Al escribir $u_h=\sum_j u_j(t)\varphi_j$ aparecen las **dos** matrices que
devuelve `assembling.m`: la de masa viene del término temporal y la de rigidez
del término espacial,

$$M\,\mathbf{u}'(t) + A\,\mathbf{u}(t) = \mathbf{b}$$

En el problema estacionario $M$ no se usa; aquí es la que carga la derivada en
el tiempo, y por eso el ensamblaje del capítulo sirve tal cual.

## El método $\theta$

Los tres esquemas son el mismo, evaluando el término $A\mathbf{u}$ como una
combinación del tiempo viejo y el nuevo:

$$M\frac{\mathbf{u}^{n+1}-\mathbf{u}^n}{\Delta t}
+ A\big(\theta\,\mathbf{u}^{n+1} + (1-\theta)\,\mathbf{u}^n\big) = \mathbf{b}$$

es decir

$$(M + \theta\,\Delta t\,A)\,\mathbf{u}^{n+1}
= (M - (1-\theta)\,\Delta t\,A)\,\mathbf{u}^n + \Delta t\,\mathbf{b}$$

| $\theta$ | esquema | orden en $\Delta t$ |
|---|---|---|
| $0$ | Euler hacia adelante (explícito) | 1 |
| $1$ | Euler hacia atrás (implícito) | 1 |
| $1/2$ | Crank–Nicolson | 2 |

El método "explícito" **también** pide resolver un sistema en cada paso, porque
$M$ no es diagonal. Lo explícito es el tratamiento del operador espacial, no el
costo del paso. (La forma usual de volverlo genuinamente explícito es condensar
la masa; no se hace aquí, pero más abajo se dice cuánto cambia la estabilidad.)

En los tres casos la matriz de la izquierda **no cambia** de un paso al otro. Se
factoriza una sola vez, fuera del ciclo, y cada paso es solo una sustitución más
un producto con la matriz de la derecha.

## Estabilidad

Si $\lambda$ son los valores propios de $A\mathbf{v}=\lambda M\mathbf{v}$, cada
modo se multiplica en cada paso por

$$g(z) = \frac{1-(1-\theta)z}{1+\theta z}, \qquad z = \Delta t\,\lambda > 0$$

Pidiendo $|g|\le 1$ se obtiene la condición

$$z\,(1-2\theta) \le 2$$

de modo que para $\theta\ge 1/2$ **no hay restricción ninguna**: el implícito y
Crank–Nicolson son incondicionalmente estables. Para $\theta<1/2$ hay que
respetar $\Delta t \le 2/((1-2\theta)\lambda_{\max})$, que con $\theta=0$ es el
$\Delta t\le 2/\lambda_{\max}$ de Euler hacia adelante. El paso lo fija el valor
propio **más grande**, que corresponde al modo más oscilatorio: justo el que la
malla resuelve peor y que no aporta nada físico.

En 1D había fórmula cerrada ($\lambda_{\max}=12/h^2$, $\Delta t^*=h^2/6$). En 2D
no: sobre esta triangulación la matriz de masa tiene estrella de 7 puntos y no
es un producto de Kronecker. Maximizando el símbolo se obtiene

$$\lambda_{\max} \longrightarrow \frac{25.856}{h^2}
\qquad\text{de donde}\qquad \Delta t^* \longrightarrow \frac{h^2}{12.93}$$

y con la masa condensada sería $\lambda_{\max}\to 8/h^2$, o sea
$\Delta t^*=h^2/4$. La masa consistente es unas 3.2 veces más restrictiva, el
mismo fenómeno que en 1D (donde el factor era exactamente 3). Como no hay
fórmula general para mallas arbitrarias, `dt_critico` calcula $\lambda_{\max}$
numéricamente.

Del otro lado del espectro, $\lambda_{\min}\to 2\pi^2$, que es la tasa de
decaimiento exacta del dato inicial que se usa aquí.

## Una advertencia sobre Crank–Nicolson

Ser incondicionalmente estable no es lo mismo que **amortiguar**. Cuando
$z\to\infty$,

$$\theta = 1 \ \text{(implícito)}: \quad g\to 0 \qquad \text{los modos rígidos se apagan}$$
$$\theta = \tfrac12 \ \text{(Crank–Nicolson)}: \quad g\to -1 \qquad \text{sobreviven, alternando signo}$$

Esa es la diferencia entre **A-estable** y **L-estable**. Con datos iniciales
suaves, como el de aquí, no se nota; con datos rugosos o discontinuos,
Crank–Nicolson con $\Delta t$ grande produce oscilaciones espurias que no decaen.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from inspect import signature
from scipy.sparse import coo_matrix
from scipy.sparse.linalg import splu, eigsh

## Entradas

Con $u_0=\sin(\pi x)\sin(\pi y)$ la solución exacta es $e^{-2\pi^2 t}u_0$: la
forma se conserva y solo decae la amplitud. En 2D la tasa es $2\pi^2$, el doble
que en 1D, así que conviene un $T$ más corto para que quede algo que medir.

In [ ]:
T   = 0.05                                      # tiempo final
u0f = lambda x, y: np.sin(np.pi*x)*np.sin(np.pi*y)
f   = lambda x, y: np.zeros_like(x)             # sin fuente: u_t = Lap(u)
uex = lambda x, y, t: np.exp(-2*np.pi**2*t)*np.sin(np.pi*x)*np.sin(np.pi*y)

## La malla y los nodos libres

Las mismas dos funciones de `MEF.ipynb`: una malla estructurada del cuadrado
partida en triángulos, y la detección topológica de la frontera (*una arista es
de frontera si pertenece a un solo triángulo*). Acá interesa el complemento: los
nodos **libres**, que son las incógnitas.

In [ ]:
def squaremesh_basico(M):
    """Malla estructurada de (0,1)^2, M subintervalos por lado.

    node: ((M+1)^2, 2)   elem: (2*M^2, 3), orientados en sentido antihorario."""
    xx = np.linspace(0, 1, M+1)
    X, Y = np.meshgrid(xx, xx, indexing='ij')
    node = np.column_stack([X.ravel(), Y.ravel()])

    elem = np.zeros((2*M*M, 3), dtype=int)
    k = 0
    for i in range(M):
        for j in range(M):
            n1 = i*(M+1) + j
            n2 = n1 + (M+1)
            n3 = n2 + 1
            n4 = n1 + 1
            elem[k]   = [n1, n2, n3]
            elem[k+1] = [n1, n3, n4]
            k += 2
    return node, elem


def nodos_libres(elem, NV):
    """Nodos que NO estan sobre la frontera."""
    total = np.vstack([elem[:, [1, 2]], elem[:, [2, 0]], elem[:, [0, 1]]])
    total = np.sort(total, axis=1)
    aristas, conteo = np.unique(total, axis=0, return_counts=True)
    bd_node = np.unique(aristas[conteo == 1])
    return np.setdiff1d(np.arange(NV), bd_node)

## Ensamblaje

`ensamblar` es `assembling.m`: mismo formato de tripletes, $3\times3=9$ entradas
por triángulo, y la identidad

$$\int_T \nabla\varphi_i\cdot\nabla\varphi_j = \frac{\mathbf{v}_i\cdot\mathbf{v}_j}{4|T|}$$

donde $\mathbf{v}_i$ es el lado opuesto al vértice $i$. La matriz de masa sale
del mismo ciclo con `area*((i==j)+1)/12`.

`lado_derecho` es solo la parte final de `ensamblar`, separada para poder
rehacer $\mathbf{b}$ en cada paso cuando la fuente depende del tiempo.

In [ ]:
def lado_derecho(node, elem, f):
    """Solo el vector b del ensamblaje (regla del baricentro)."""
    NV = len(node)
    v1 = node[elem[:, 1]] - node[elem[:, 0]]
    v2 = node[elem[:, 2]] - node[elem[:, 0]]
    area = 0.5*np.abs(v1[:, 0]*v2[:, 1] - v1[:, 1]*v2[:, 0])
    bar = node[elem].mean(axis=1)                      # baricentros
    bt = area*f(bar[:, 0], bar[:, 1])/3
    bt = np.column_stack([bt, bt, bt])
    return np.bincount(elem.ravel(order='F'), weights=bt.ravel(order='F'), minlength=NV)


def ensamblar(node, elem, f):
    """Rigidez, masa y lado derecho P1 en 2D. Es assembling.m."""
    NV, NT = len(node), len(elem)

    # ve[:,:,i] = lado opuesto al vertice i
    ve = np.zeros((NT, 2, 3))
    ve[:, :, 2] = node[elem[:, 1]] - node[elem[:, 0]]
    ve[:, :, 0] = node[elem[:, 2]] - node[elem[:, 1]]
    ve[:, :, 1] = node[elem[:, 0]] - node[elem[:, 2]]
    area = 0.5*np.abs(-ve[:, 0, 2]*ve[:, 1, 1] + ve[:, 1, 2]*ve[:, 0, 1])

    ii, jj, sA, sM = [], [], [], []
    for i in range(3):
        for j in range(3):
            ii.append(elem[:, i])
            jj.append(elem[:, j])
            sA.append(np.sum(ve[:, :, i]*ve[:, :, j], axis=1)/(4*area))
            sM.append(area*((i == j) + 1)/12)
    ii = np.concatenate(ii); jj = np.concatenate(jj)
    A = coo_matrix((np.concatenate(sA), (ii, jj)), shape=(NV, NV)).tocsr()
    M = coo_matrix((np.concatenate(sM), (ii, jj)), shape=(NV, NV)).tocsr()
    A.eliminate_zeros()                       # scipy guarda los ceros exactos

    return A, M, lado_derecho(node, elem, f), area

## El método $\theta$

El lado derecho lleva **la misma combinación** que $A\mathbf{u}$, y no es un
capricho. Integrando el sistema sobre $[t_n, t_{n+1}]$ queda

$$M(\mathbf{u}^{n+1}-\mathbf{u}^n) + A\!\int \mathbf{u}\,dt = \int \mathbf{b}\,dt$$

y el método $\theta$ es elegir una cuadratura para esas dos integrales. Con
$\theta=1/2$ la de la izquierda es la regla del trapecio, de error local
$O(\Delta t^3)$. Si la de la derecha se aproximara por el rectángulo
$\Delta t\,\mathbf{b}^{n+1}$, que es solo $O(\Delta t^2)$, mandaría el término
más débil y Crank–Nicolson caería a orden 1. La sección 8 lo mide.

Con $f$ estacionaria $\mathbf{b}^{n+1}=\mathbf{b}^n$ y la combinación se reduce
a $\mathbf{b}$, que es el caso de casi todo este archivo. La fuente puede darse
de dos formas y se detecta sola, mirando cuántos argumentos declara:

```
f = lambda x, y:     ...    estacionaria
f = lambda x, y, t:  ...    dependiente del tiempo
```

In [ ]:
def resolver_theta(node, elem, dt, T, u0f, f, theta):
    """Metodo theta:

    (M + theta dt A) u^{n+1}
        = (M - (1-theta) dt A) u^n + dt [theta b^{n+1} + (1-theta) b^n]

    theta = 0 explicito, 1 implicito, 1/2 Crank-Nicolson."""
    NV = len(node)
    free = nodos_libres(elem, NV)

    # cuantos argumentos declara f dice si la fuente varia en el tiempo
    varia = len(signature(f).parameters) == 3
    cero = lambda x, y: np.zeros_like(x)
    A, M, b, _ = ensamblar(node, elem, cero if varia else f)

    u = u0f(node[:, 0], node[:, 1]).astype(float)
    u[np.setdiff1d(np.arange(NV), free)] = 0.0        # Dirichlet homogeneo
    if T == 0:
        return u

    nsteps = max(1, int(round(T/dt)))
    dt = T/nsteps                        # ajusta dt para caer exactamente en T

    # Ninguna de las dos matrices depende del paso: se arman una sola vez.
    Mf = M[free][:, free].tocsr()
    Af = A[free][:, free].tocsr()
    lu = splu((Mf + theta*dt*Af).tocsc())
    R  = (Mf - (1-theta)*dt*Af).tocsr()

    uf = u[free]
    if varia:
        b_nuevo = lado_derecho(node, elem, lambda x, y: f(x, y, 0.0))

    for k in range(nsteps):
        if varia:
            # el b^{n+1} del paso anterior es el b^n de este: se reaprovecha
            b_viejo = b_nuevo
            tk = (k+1)*dt
            b_nuevo = lado_derecho(node, elem, lambda x, y: f(x, y, tk))
            bk = theta*b_nuevo + (1-theta)*b_viejo
        else:
            bk = b
        uf = lu.solve(R @ uf + dt*bk[free])

    u[free] = uf
    return u


def resolver_BE(node, elem, dt, T, u0f, f):
    """Euler hacia atras (theta = 1). Incondicionalmente estable."""
    return resolver_theta(node, elem, dt, T, u0f, f, 1.0)


def resolver_CN(node, elem, dt, T, u0f, f):
    """Crank-Nicolson (theta = 1/2). Orden 2 en el tiempo."""
    return resolver_theta(node, elem, dt, T, u0f, f, 0.5)


def resolver_FE(node, elem, dt, T, u0f, f):
    """Euler hacia adelante (theta = 0). Estable solo si dt <= dt_critico."""
    return resolver_theta(node, elem, dt, T, u0f, f, 0.0)

## Los valores propios y el error en $L^2$

`eigsh` resuelve el problema generalizado $A\mathbf{v}=\lambda M\mathbf{v}$.
Para el **mayor** basta pedir `which='LA'`; para el **menor** conviene el
desplazamiento `sigma=0` (*shift-invert*), que lo encuentra en pocas
iteraciones en vez de converger lentísimo.

In [ ]:
def lambda_min(node, elem):
    """Menor valor propio de A v = lambda M v.

    Es la tasa de decaimiento del modo mas lento de la solucion discreta, el
    analogo de 2*pi^2 en el continuo. Tiende a 2*pi^2 + O(h^2) POR ARRIBA."""
    A, M, _, _ = ensamblar(node, elem, lambda x, y: np.zeros_like(x))
    free = nodos_libres(elem, len(node))
    return eigsh(A[free][:, free].tocsc(), k=1, M=M[free][:, free].tocsc(),
                 sigma=0.0, which='LM', return_eigenvectors=False)[0]


def dt_critico(node, elem, theta):
    """Paso maximo admisible del metodo theta.

    De |g| <= 1 sale dt*lambda_max*(1-2 theta) <= 2. Para theta >= 1/2 no hay
    restriccion. En esta malla lambda_max -> 25.856/h^2, de donde el paso
    critico del explicito tiende a h^2/12.93."""
    if theta >= 0.5:
        return np.inf
    A, M, _, _ = ensamblar(node, elem, lambda x, y: np.zeros_like(x))
    free = nodos_libres(elem, len(node))
    lmax = eigsh(A[free][:, free].tocsc(), k=1, M=M[free][:, free].tocsc(),
                 which='LA', return_eigenvectors=False)[0]
    return 2/((1 - 2*theta)*lmax)


def error_L2(node, elem, u, uexT):
    """Norma L2 del error, con la regla de grado 5 (7 puntos) del capitulo."""
    a1, b1 = 0.470142064105115, 0.059715871789770
    a2, b2 = 0.101286507323456, 0.797426985353087
    L = np.array([[1/3, 1/3, 1/3],
                  [b1, a1, a1], [a1, b1, a1], [a1, a1, b1],
                  [b2, a2, a2], [a2, b2, a2], [a2, a2, b2]])
    W = np.array([9/40] + [0.132394152788506]*3 + [0.125939180544827]*3)

    _, _, _, area = ensamblar(node, elem, lambda x, y: np.zeros_like(x))
    e2 = 0.0
    for q in range(len(W)):
        # punto de cuadratura y valor de u_h ahi, en coordenadas baricentricas
        xq = sum(L[q, k]*node[elem[:, k], 0] for k in range(3))
        yq = sum(L[q, k]*node[elem[:, k], 1] for k in range(3))
        uq = sum(L[q, k]*u[elem[:, k]]       for k in range(3))
        e2 += np.sum(W[q]*(uq - uexT(xq, yq))**2 * area)
    return np.sqrt(e2)

## 1. Una corrida

In [ ]:
n  = 32                                # subdivisiones por lado
dt = T/200                             # paso de tiempo

node, elem = squaremesh_basico(n)
ue = uex(node[:, 0], node[:, 1], T)

print(f'n = {n}  ->  {len(node)} nodos, {len(elem)} triangulos')
print(f'dt = {dt:.2e} ({round(T/dt)} pasos);  '
      f'dt critico del explicito: {dt_critico(node, elem, 0):.3e}')
print('\n   metodo            err_max       err_L2')
for nombre, th in [('Euler atras', 1.0), ('Crank-Nicolson', 0.5)]:
    uk = resolver_theta(node, elem, dt, T, u0f, f, th)
    print(f'   {nombre:<16}  {np.linalg.norm(uk-ue, np.inf):.4e}   '
          f'{error_L2(node, elem, uk, lambda x, y: uex(x, y, T)):.4e}')

Cuidado al leer esa tabla: Euler hacia atrás sale con **menos** error que
Crank–Nicolson, y no es que sea mejor. Su error temporal y el espacial tienen
signos opuestos —el esquema decae de menos, la malla decae de más— y para este
$\Delta t$ se cancelan de casualidad. El error de Crank–Nicolson, en cambio, ya
es prácticamente el error espacial puro, que es la precisión real de la
discretización. La cancelación desaparece al cambiar $\Delta t$ o la malla; la
sección 4 lo muestra con claridad.

## 2. Evolución en el tiempo

In [ ]:
instantes = [0, 0.005, 0.02, 0.05]

fig = plt.figure(figsize=(10, 8))
for k, tk in enumerate(instantes):
    uk = resolver_CN(node, elem, dt, tk, u0f, f)
    ax = fig.add_subplot(2, 2, k+1, projection='3d')
    # misma escala de color en los cuatro: asi el decaimiento tambien se ve
    s = ax.plot_trisurf(node[:, 0], node[:, 1], uk, triangles=elem,
                        cmap='viridis', vmin=0, vmax=1, linewidth=0)
    ax.set_zlim(0, 1); ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.set_title(f't = {tk:g}')
fig.suptitle('Crank-Nicolson')
fig.colorbar(s, ax=fig.axes, shrink=0.55, pad=0.08)
plt.show()

## 3. Convergencia en el tiempo

Para ver el orden temporal limpio **no** se compara contra la solución exacta,
sino contra una referencia calculada en la **misma malla** con un $\Delta t$
diminuto. Así el error espacial, que es idéntico en las dos, se cancela
exactamente, y lo que queda es puro error temporal. Es la única forma de ver el
orden 2 de Crank–Nicolson: contra la solución exacta su error cae por debajo del
error espacial de inmediato y la medición se satura.

In [ ]:
uref = resolver_CN(node, elem, T/40000, T, u0f, f)

nsT = np.array([10, 20, 40, 80, 160, 320])
dtT = T/nsT
errBEt = np.zeros(len(nsT)); errCNt = np.zeros(len(nsT))

print(f'===== ORDEN EN EL TIEMPO  (n = {n}, referencia: CN con dt = T/40000) =====')
print('   pasos      dt       err BE    orden     err CN    orden')
for k in range(len(nsT)):
    errBEt[k] = np.linalg.norm(resolver_BE(node, elem, dtT[k], T, u0f, f) - uref, np.inf)
    errCNt[k] = np.linalg.norm(resolver_CN(node, elem, dtT[k], T, u0f, f) - uref, np.inf)
    o1 = np.log2(errBEt[k-1]/errBEt[k]) if k else np.nan
    o2 = np.log2(errCNt[k-1]/errCNt[k]) if k else np.nan
    print(f'{nsT[k]:8d}  {dtT[k]:.2e}  {errBEt[k]:.3e}  {o1:6.2f}   {errCNt[k]:.3e}  {o2:6.2f}')

El explícito solo existe por debajo de $\Delta t^*$, muy a la izquierda de ese
rango.

In [ ]:
dtc  = dt_critico(node, elem, 0)
nsFE = np.array([1000, 2000, 4000])
dtFE = T/nsFE
errFEt = np.array([np.linalg.norm(resolver_FE(node, elem, d, T, u0f, f) - uref, np.inf)
                   for d in dtFE])

print(f'Euler explicito (solo donde es estable, dt* = {dtc:.2e}):')
for ns, d, e in zip(nsFE, dtFE, errFEt):
    print(f'{ns:8d}  {d:.2e}  {e:.3e}')

### La predicción teórica, exacta para este dato inicial

El dato inicial es un modo propio, así que en la malla se propaga como un solo
modo discreto: la solución semidiscreta es $e^{-\lambda_h t}\mathbf{v}_h$ y el
método $\theta$ la reemplaza por $g(\Delta t\lambda_h)^n\,\mathbf{v}_h$. El
error es entonces

$$\big|\,g(\Delta t\,\lambda_h)^n - e^{-\lambda_h T}\,\big|$$

**sin constantes libres**. Desarrollando para $\Delta t$ chico se recuperan los
términos de libro, con $\lambda=\lambda_h$:

$$\text{Euler atrás:}\quad e \approx e^{-\lambda T}\,\frac{\lambda^2 \Delta t\, T}{2}
\qquad\qquad
\text{Crank–Nicolson:}\quad e \approx e^{-\lambda T}\,\frac{\lambda^3 \Delta t^2 T}{12}$$

Ojo que ahí va $\lambda_h$ y no $2\pi^2$: usar el valor continuo deja un sesgo
del orden de $(\lambda_h/2\pi^2)^3-1$, que en esta malla es medio por ciento.

In [ ]:
lh  = lambda_min(node, elem)
gBE = lambda z: 1/(1+z)
gCN = lambda z: (1 - z/2)/(1 + z/2)
gFE = lambda z: 1 - z

teoBEt = np.abs(gBE(dtT*lh)**nsT  - np.exp(-lh*T))
teoCNt = np.abs(gCN(dtT*lh)**nsT  - np.exp(-lh*T))
teoFEt = np.abs(gFE(dtFE*lh)**nsFE - np.exp(-lh*T))

print(f'lambda_h = {lh:.6f}  (2*pi^2 = {2*np.pi**2:.6f})')
print(f'razon teoria/medicion:  BE en [{(teoBEt/errBEt).min():.4f}, {(teoBEt/errBEt).max():.4f}],'
      f'  CN en [{(teoCNt/errCNt).min():.4f}, {(teoCNt/errCNt).max():.4f}]')

## 4. Convergencia en el espacio

Ahora contra la solución exacta, con $\Delta t$ **fijo**. Crank–Nicolson exhibe
el orden 2 en $h$ sin problema; Euler hacia atrás no puede, porque su error
temporal se queda constante mientras el espacial baja, y los órdenes medidos se
vuelven absurdos (llegan a dar negativos cuando los dos errores, que tienen
signos opuestos, se cancelan). Ese contraste es la razón práctica para preferir
un esquema de orden 2 en el tiempo.

In [ ]:
nsH = np.array([4, 8, 16, 32, 64])
hH  = 1/nsH
errBEh = np.zeros(len(nsH)); errCNh = np.zeros(len(nsH))

print('===== ORDEN EN EL ESPACIO  (dt = T/200 fijo) =====')
print('    n       h       err BE    orden     err CN    orden')
for k, nk in enumerate(nsH):
    nd, el = squaremesh_basico(nk)
    uek = uex(nd[:, 0], nd[:, 1], T)
    errBEh[k] = np.linalg.norm(resolver_BE(nd, el, T/200, T, u0f, f) - uek, np.inf)
    errCNh[k] = np.linalg.norm(resolver_CN(nd, el, T/200, T, u0f, f) - uek, np.inf)
    o1 = np.log2(errBEh[k-1]/errBEh[k]) if k else np.nan
    o2 = np.log2(errCNh[k-1]/errCNh[k]) if k else np.nan
    print(f'{nk:5d}  {hH[k]:.4f}  {errBEh[k]:.3e}  {o1:6.2f}   {errCNh[k]:.3e}  {o2:6.2f}')

### Por qué la curva de Euler hacia atrás hace una "V"

En el gráfico de la sección 5 la curva azul baja, toca un mínimo y vuelve a
subir. No es un error numérico: es la cancelación de arriba, vista de cerca.
Basta mirar el error **con signo** en el nodo central, donde es máximo.

In [ ]:
print(f'{"n":>4} {"h":>8} {"lambda_h":>10} {"err BE":>12} {"err CN":>12} {"BE - CN":>12}')
for nk in nsH:
    nd, el = squaremesh_basico(nk)
    uek = uex(nd[:, 0], nd[:, 1], T)
    c = np.argmax(uek)                            # el nodo central
    uB = resolver_BE(nd, el, T/200, T, u0f, f)
    uC = resolver_CN(nd, el, T/200, T, u0f, f)
    print(f'{nk:4d} {1/nk:8.4f} {lambda_min(nd, el):10.4f} '
          f'{uB[c]-uek[c]:12.3e} {uC[c]-uek[c]:12.3e} {uB[c]-uC[c]:12.3e}')

# la prediccion del termino temporal de BE, que no depende de h
lam = 2*np.pi**2
print(f'\ntermino temporal de BE previsto: exp(-lam T) lam^2 dt T/2 = '
      f'{np.exp(-lam*T)*lam**2*(T/200)*T/2:.3e}')

La última columna es **constante**: ese es el error temporal de Euler hacia
atrás, que no depende de $h$ y coincide con la fórmula de abajo. Crank–Nicolson,
con $\Delta t$ tan chico, ya no tiene error temporal visible, así que su columna
es el error espacial puro: **negativo** (porque $\lambda_h > 2\pi^2$, la malla
hace decaer la solución de más) y bajando como $h^2$.

Euler hacia atrás es esa misma curva más una constante **positiva** (decae de
menos, porque $1/(1+z) > e^{-z}$). Al refinar, la parte negativa se encoge
mientras la positiva no se mueve: en algún punto la suma cruza por cero, y ahí
está el pico hacia abajo. Pasado ese punto la curva vuelve a subir hasta
estacionarse en el valor del error temporal, del que ya no baja por más que se
refine la malla.

Dos moralejas: el mínimo no es que el método mejore, y la pendiente 2 que se le
ve a la izquierda del pico es ilusoria — se está midiendo la suma de dos cosas
que apuntan en direcciones contrarias.

## 5. Gráficos del error

Los símbolos son lo medido; las líneas, la predicción teórica de la sección 3.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

azul, rojo, amar = '#0072BD', '#D95319', '#EDB120'
ax1.loglog(dtT,  teoBEt, '-', color=azul, lw=1.2, label='teoria (modo unico)')
ax1.loglog(dtT,  teoCNt, '-', color=rojo, lw=1.2)
ax1.loglog(dtFE, teoFEt, '-', color=amar, lw=1.2)
ax1.loglog(dtT,  errBEt, 'o', ms=7, mfc='none', color=azul, label='Euler atras')
ax1.loglog(dtT,  errCNt, 's', ms=7, mfc='none', color=rojo, label='Crank-Nicolson')
ax1.loglog(dtFE, errFEt, '^', ms=7, mfc='none', color=amar, label='Euler adelante')
ax1.axvline(dtc, color='r', lw=1)
ax1.text(dtc, ax1.get_ylim()[0], ' $dt^*$', color='r', va='bottom')
ax1.set_xlabel('dt'); ax1.set_ylabel('error temporal')
ax1.set_title(f'Error vs dt  (n = {n}):  pendientes 1 y 2')
ax1.legend(loc='lower right'); ax1.grid(True, which='both', alpha=0.3)

ax2.loglog(hH, errBEh, 'o-', lw=1.4, label='Euler atras')
ax2.loglog(hH, errCNh, 's-', lw=1.4, label='Crank-Nicolson')
ax2.loglog(hH, 0.35*errCNh[0]*(hH/hH[0])**2, 'k:', label='pendiente 2')
ax2.set_xlabel('h'); ax2.set_ylabel('error total')
ax2.set_title('Error vs h  (dt = T/200 fijo)')
ax2.legend(loc='lower right'); ax2.grid(True, which='both', alpha=0.3)
plt.show()

## 6. La restricción de estabilidad del explícito

Misma malla y mismo $\Delta t$ para los tres. Los dos incondicionalmente
estables funcionan en toda la columna; el explícito solo por debajo de
$\Delta t^*$, y apenas se pasa no se degrada: **explota**.

In [ ]:
ne = 16
nodeE, elemE = squaremesh_basico(ne)
dtcE = dt_critico(nodeE, elemE, 0)
he   = 1/ne
ueE  = uex(nodeE[:, 0], nodeE[:, 1], T)

print(f'===== ESTABILIDAD  (n = {ne}, h = {he:.4f}) =====')
print(f'dt* = 2/lambda_max = {dtcE:.3e}   (h^2/12.93 = {he**2/12.93:.3e}, '
      f'condensada h^2/4 = {he**2/4:.3e})')
print('   pasos      dt      dt/dt*    err atras     err CN     err adelante')
for ns in [50, 100, 200, 1000]:
    dtk = T/ns
    e1 = np.linalg.norm(resolver_BE(nodeE, elemE, dtk, T, u0f, f) - ueE, np.inf)
    e2 = np.linalg.norm(resolver_CN(nodeE, elemE, dtk, T, u0f, f) - ueE, np.inf)
    e3 = np.linalg.norm(resolver_FE(nodeE, elemE, dtk, T, u0f, f) - ueE, np.inf)
    print(f'{ns:8d}  {dtk:.3e}  {dtk/dtcE:6.2f}   {e1:.3e}   {e2:.3e}   {e3:.3e}')

## 7. Paso enorme

Un solo paso que cubre todo el intervalo. El explícito explotaría sin remedio;
los otros dos siguen acotados, apenas imprecisos.

In [ ]:
print(f'===== PASO ENORME  (n = {n}) =====')
print('   pasos      dt      err atras    err CN')
for ns in [1, 2, 5, 200]:
    e1 = np.linalg.norm(resolver_BE(node, elem, T/ns, T, u0f, f) - ue, np.inf)
    e2 = np.linalg.norm(resolver_CN(node, elem, T/ns, T, u0f, f) - ue, np.inf)
    print(f'{ns:8d}  {T/ns:.3e}  {e1:.3e}  {e2:.3e}')

print(f'En esta malla el explicito pediria dt <= {dtc:.2e}, o sea al menos')
print(f'{int(np.ceil(T/dtc))} pasos para llegar a T = {T:g}.')

## 8. Fuente dependiente del tiempo

Comprobación de que el promedio del lado derecho conserva el orden 2. Se fabrica
una solución que no es la del decaimiento natural,

$$u = e^{-t}\sin(\pi x)\sin(\pi y) \qquad\Longrightarrow\qquad
f = (2\pi^2-1)\,e^{-t}\sin(\pi x)\sin(\pi y)$$

de modo que $f$ sí dependa del tiempo. Si el lado derecho se evaluara solo en
$t^{n+1}$, en vez de promediarlo, Crank–Nicolson bajaría a orden 1.

In [ ]:
ft  = lambda x, y, t: (2*np.pi**2 - 1)*np.exp(-t)*np.sin(np.pi*x)*np.sin(np.pi*y)
u0t = lambda x, y: np.sin(np.pi*x)*np.sin(np.pi*y)

reft = resolver_CN(node, elem, T/40000, T, u0t, ft)      # referencia, misma malla

print(f'===== FUENTE DEPENDIENTE DEL TIEMPO  (n = {n}) =====')
print('   pasos      dt       err CN     orden')
ep = np.nan
for ns in [10, 20, 40, 80]:
    e = np.linalg.norm(resolver_CN(node, elem, T/ns, T, u0t, ft) - reft, np.inf)
    print(f'{ns:8d}  {T/ns:.2e}  {e:.3e}  {np.log2(ep/e):7.2f}')
    ep = e

uexT = lambda x, y: np.exp(-T)*np.sin(np.pi*x)*np.sin(np.pi*y)
err = np.linalg.norm(resolver_CN(node, elem, T/200, T, u0t, ft)
                     - uexT(node[:, 0], node[:, 1]), np.inf)
print(f'error contra la solucion exacta con dt = T/200: {err:.4e}')